# Sistema Híbrido de Deep Learning para Predicción QQQ
## Prototipo Funcional — Universidad del Quindío

**Objetivo:** Entrenar y evaluar el sistema completo `HybridPredictiveModel`
(BiLSTM + CrossAttention + FinBERT) con walk-forward validation, produciendo
métricas de RMSE, MAE, Directional Accuracy y Sharpe Ratio para horizontes **t+1 y t+5**.

**Este notebook es el prototipo funcional de la tesis** — no es el prototipo de póster.
Diferencias clave vs. `QQQ_Prototipo_Colab.ipynb`:

| Aspecto | Póster | Este notebook |
|---------|--------|---------------|
| Modelo | `LSTMDirectionModel` (inline) | `HybridPredictiveModel` (desde `src/`) |
| Target | Binario sube/baja | Regresión retorno % (t+1 **y** acumulado 5 días) |
| Validación | Split estático 70/15/15 | Walk-forward 5 folds |
| Métricas | Accuracy, AUC-ROC | RMSE, MAE, DA, Sharpe, MaxDD |
| Generativo | No | TimeGAN + WGAN-GP (Sección 6) |
| Sentimiento | No | FinBERT (zeros si sin corpus) |

**Secciones:**
0. Setup & Bootstrap
1. Pipeline de Datos
2. EDA Rápido
3. Walk-Forward Training
4. Evaluación Test Out-of-Sample
5. Backtesting
6. TimeGAN — Módulo Generativo *(opcional)*
7. Resumen Final

## 0. Setup — Bootstrap Colab
Ejecuta esta celda primero. Clona o actualiza el repo y carga credenciales.

In [ ]:
import os, sys, shutil

GITHUB_REPO = 'https://github.com/JuliRey2000/HybridsistemQQQ.git'
REPO_DIR    = '/content/HybridsistemQQQ'
IN_COLAB    = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {GITHUB_REPO} {REPO_DIR}')
        print(f'Repo clonado en {REPO_DIR}')
    else:
        os.system(f'git -C {REPO_DIR} pull')
        print(f'Repo actualizado en {REPO_DIR}')
        # Limpiar bytecode cacheado para forzar re-import del código actualizado
        pycache = os.path.join(REPO_DIR, 'src', '__pycache__')
        if os.path.exists(pycache):
            shutil.rmtree(pycache)
            print('  __pycache__ limpiado (fuerza re-import de src/)')
    os.chdir(REPO_DIR)

    try:
        from google.colab import userdata
        os.environ['MLFLOW_TRACKING_USERNAME'] = 'JuliRey2000'
        os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('DAGSHUB_TOKEN')
        print('Credenciales DagsHub cargadas desde Colab Secrets.')
    except Exception:
        print('AVISO: agrega DAGSHUB_TOKEN en Colab Secrets (sidebar → candado).')

# Asegurar que src/ sea importable desde el directorio activo
cwd = os.getcwd()
if cwd not in sys.path:
    sys.path.insert(0, cwd)

print(f'Directorio activo: {cwd}')
print(f'En Colab        : {IN_COLAB}')

In [ ]:
%%capture
!pip install --upgrade --quiet yfinance ta scikit-learn seaborn tqdm mlflow dagshub statsmodels
print('Dependencias instaladas.')

In [ ]:
import warnings
import logging
import random
import copy
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import joblib
import mlflow
import dagshub

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# ── Semilla global (CRISP-ML(Q) — reproducibilidad) ─────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')
if device == 'cuda':
    print(f'GPU       : {torch.cuda.get_device_name(0)}')
    print(f'VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Sin GPU. Entrenamiento en CPU (más lento). Activa GPU en Colab: Entorno → T4 GPU')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 11
sns.set_style('whitegrid')

# ── Importar módulos del proyecto ────────────────────────────────────────────
from src.data_pipeline import DataPipeline
from src.models import (
    HybridPredictiveModel, TimeGANGenerator, WassersteinCritic, count_parameters
)
from src.train import Trainer, GANTrainer, make_dataloader, predict, generate_scenarios
from src.utils import (
    walk_forward_splits, final_test_split,
    scale_price_sequences, transform_price_sequences,
    rmse, mae, directional_accuracy, sharpe_ratio, sortino_ratio, max_drawdown,
    predictive_metrics, long_short_strategy,
    plot_predictions, plot_training_history,
    plot_cumulative_returns, plot_generated_scenarios, generative_metrics,
)
print('Imports OK')

In [ ]:
DAGSHUB_USER = 'JuliRey2000'
DAGSHUB_REPO = 'HybridsistemQQQ'

dagshub.init(repo_owner=DAGSHUB_USER, repo_name=DAGSHUB_REPO, mlflow=True)
mlflow.set_experiment('QQQ-HybridPredictive')

print(f'MLflow tracking URI : {mlflow.get_tracking_uri()}')
print(f'Experimento activo  : QQQ-HybridPredictive')
print(f'Ver runs en         : https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow')

## 1. Pipeline de Datos

Descarga QQQ desde Yahoo Finance (2015-2024), calcula 9 indicadores técnicos + VIX
(**10 features** en total) y empaqueta en ventanas deslizantes de 30 días. Si existe
`data/processed/finbert_embeddings.csv`, lo carga para la rama de sentimiento;
de lo contrario usa vectores de ceros (placeholder).

**Targets:** `y_t1` es el retorno log diario (%) de t+1; `y_t5` es el **retorno
acumulado** de 5 días — suma de los 5 log-returns, equivalente a `100·ln(P_{t+5}/P_t)`.
(Hasta 2026-06-12 `y_t5` era el retorno puntual del día t+5, con señal casi nula a
5 días vista; su RMSE vive ahora en escala ~√5× la diaria.)

Las secuencias se guardan **sin normalizar**: la normalización (StandardScaler) se
aplica por fold en la Sección 3, ajustada solo con datos de entrenamiento para
evitar data leakage.

Si ya ejecutaste el pipeline antes, la celda carga desde disco directamente.

In [ ]:
# ── Parámetros globales ──────────────────────────────────────────────────────
TICKER     = 'QQQ'
START_DATE = '2015-01-01'
END_DATE   = '2024-12-31'
LOOKBACK   = 30
PROCESSED  = 'data/processed'

# ── Cargar desde disco si ya existe (evita re-descargar) ─────────────────────
if (Path(PROCESSED) / 'price_seqs.npy').exists():
    print('Cargando datos procesados desde disco...')
    data = {
        'price_seqs': np.load(f'{PROCESSED}/price_seqs.npy'),
        'sentiments': np.load(f'{PROCESSED}/sentiments.npy'),
        'y_t1':       np.load(f'{PROCESSED}/y_t1.npy'),
        'y_t5':       np.load(f'{PROCESSED}/y_t5.npy'),
        'dates':      np.load(f'{PROCESSED}/dates.npy', allow_pickle=True),
    }
else:
    print('Ejecutando pipeline de datos (primera vez — descarga ~10 años QQQ)...')
    pipeline = DataPipeline(
        ticker=TICKER,
        start_date=START_DATE,
        end_date=END_DATE,
        lookback=LOOKBACK,
        sentiment_path=f'{PROCESSED}/finbert_embeddings.csv',  # carga si existe
        save_dir=PROCESSED,
    )
    data = pipeline.run()

# ── Guard: caché generada antes de añadir VIX debe regenerarse ───────────────
if data['price_seqs'].shape[2] < 10:
    raise RuntimeError(
        f"Caché obsoleta: price_seqs tiene {data['price_seqs'].shape[2]} features "
        f"(se esperan 10 = 9 indicadores técnicos + VIX_Close). "
        f"Borra data/processed/*.npy y reejecuta esta celda."
    )

# ── Guard: caché con y_t5 puntual (definición antigua) debe regenerarse ──────
# y_t5 acumulado (suma de 5 log-returns) tiene std ~√5 ≈ 2.2× la de y_t1;
# si la razón es ~1, la caché es anterior al cambio de definición del target
if data['y_t5'].std() < 1.5 * data['y_t1'].std():
    raise RuntimeError(
        f"Caché obsoleta: std(y_t5)={data['y_t5'].std():.3f} ≈ std(y_t1)="
        f"{data['y_t1'].std():.3f} — y_t5 parece retorno puntual del día t+5, "
        f"no acumulado de 5 días. Borra data/processed/*.npy y reejecuta esta celda."
    )

# ── Dimensiones ──────────────────────────────────────────────────────────────
N_SAMPLES  = len(data['y_t1'])
N_FEATURES = data['price_seqs'].shape[2]   # 10 = 9 indicadores técnicos + VIX_Close
SENT_DIM   = data['sentiments'].shape[1]   # 768 (FinBERT CLS dim)

has_sentiment = bool(np.any(data['sentiments'] != 0))

print(f'\n{"="*55}')
print(f'  price_seqs  : {data["price_seqs"].shape}  (n, lookback, features)')
print(f'  sentiments  : {data["sentiments"].shape}  ({"REAL FinBERT" if has_sentiment else "ceros — sin corpus FinBERT"})')
print(f'  y_t1        : {data["y_t1"].shape}')
print(f'  y_t5        : {data["y_t5"].shape}  (retorno acumulado 5 días)')
print(f'  N_FEATURES  : {N_FEATURES}')
print(f'{"="*55}')

if not has_sentiment:
    print('\n[INFO] Sentimiento = ceros. El modelo usará solo la rama de precios.')
    print('       Para activar FinBERT: construir corpus con FNSPID + Tiingo y')
    print('       guardar embeddings en data/processed/finbert_embeddings.csv')

## 2. EDA Rápido

In [ ]:
price_df = pd.read_csv(f'{PROCESSED}/price_df.csv', index_col=0, parse_dates=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle(f'EDA — {TICKER} {START_DATE[:4]}–{END_DATE[:4]}', fontsize=14, fontweight='bold')

ax = axes[0, 0]
ax.plot(price_df.index, price_df['Close'], color='steelblue', linewidth=0.8)
ax.axvspan(pd.Timestamp('2020-02-19'), pd.Timestamp('2020-03-23'),
           alpha=0.25, color='red', label='COVID crash')
ax.set_title('Precio de Cierre QQQ')
ax.set_ylabel('USD')
ax.legend(fontsize=9)

ax = axes[0, 1]
ax.plot(price_df.index, price_df['Daily_Return'], color='gray', linewidth=0.5, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Retorno Diario Logarítmico (%)')
ax.set_ylabel('%')

ax = axes[1, 0]
ax.hist(price_df['Daily_Return'], bins=80, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(price_df['Daily_Return'].mean(), color='red', linestyle='--',
           label=f'Media: {price_df["Daily_Return"].mean():.3f}%')
ax.set_title('Distribución de Retornos')
ax.set_xlabel('%')
ax.legend(fontsize=9)

ax = axes[1, 1]
ax.plot(price_df.index, price_df['RSI_14'], color='purple', linewidth=0.7)
ax.axhline(70, color='red', linestyle='--', linewidth=0.8, label='Sobrecomprado (70)')
ax.axhline(30, color='green', linestyle='--', linewidth=0.8, label='Sobrevendido (30)')
ax.set_title('RSI 14')
ax.set_ylim(0, 100)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('eda_overview.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'Días de mercado  : {len(price_df)}')
print(f'Retorno medio    : {price_df["Daily_Return"].mean():.4f}%')
print(f'Retorno std      : {price_df["Daily_Return"].std():.4f}%')
print(f'Skewness         : {price_df["Daily_Return"].skew():.4f}')
print(f'Kurtosis (exc.)  : {price_df["Daily_Return"].kurtosis():.4f}  (> 3 = colas pesadas)')

# Verificación COVID
covid = price_df.loc['2020-03-01':'2020-04-30', 'Daily_Return']
print(f'COVID min (mar-abr 2020): {covid.min():.2f}%  ✓  (crisis retenida sin modificación)')

In [ ]:
# ── EDA estadístico: estacionariedad, autocorrelación y correlación ──────────
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

returns = price_df['Daily_Return'].dropna()

# Test ADF (H0: raíz unitaria → serie NO estacionaria)
print('=== Test ADF de estacionariedad ===')
for nombre, serie in [('Daily_Return', returns), ('Close', price_df['Close'].dropna())]:
    stat, pval, *_ = adfuller(serie.values)
    veredicto = 'ESTACIONARIA (rechaza H0)' if pval < 0.05 else 'NO estacionaria'
    print(f'  {nombre:13s}: stat={stat:8.3f}  p-value={pval:.2e}  → {veredicto}')
print('  (Esperado: retornos estacionarios, precio no — justifica modelar retornos)')

# ACF/PACF — retornos (memoria lineal) y |retornos| (volatility clustering)
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
plot_acf(returns, lags=40, ax=axes[0, 0], title='ACF — retornos')
plot_pacf(returns, lags=40, ax=axes[0, 1], method='ywm', title='PACF — retornos')
plot_acf(returns.abs(), lags=40, ax=axes[1, 0], title='ACF — |retornos| (volatility clustering)')
plot_pacf(returns.abs(), lags=40, ax=axes[1, 1], method='ywm', title='PACF — |retornos|')
plt.tight_layout()
plt.savefig('eda_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Lectura: ACF de retornos ≈ 0 (poca memoria lineal → modelos lineales no bastan);')
print('ACF de |retornos| significativa varios lags → volatility clustering (hecho estilizado).')

# Matriz de correlación de los 10 features que entran al modelo
FEATURE_COLS = [c for c in price_df.columns
                if c not in ['Daily_Return', 'Close', 'Open', 'High', 'Low', 'Volume']]
corr = price_df[FEATURE_COLS].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, cbar_kws={'shrink': 0.8})
plt.title(f'Correlación entre los {len(FEATURE_COLS)} features del modelo',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

print('Pares con |correlación| > 0.9 (candidatos a redundancia):')
pares = [(a, b, corr.loc[a, b]) for i, a in enumerate(FEATURE_COLS)
         for b in FEATURE_COLS[i+1:] if abs(corr.loc[a, b]) > 0.9]
for a, b, c in pares:
    print(f'  {a:12s} ↔ {b:12s}: {c:+.3f}')
if not pares:
    print('  (ninguno)')

## 3. Walk-Forward Training — HybridPredictiveModel

Validación cronológica con 5 folds. El último 15% se reserva como test out-of-sample
(nunca visto durante entrenamiento ni validación). En cada fold el conjunto de
entrenamiento crece acumulativamente — simula uso real en producción.

**Normalización sin leakage:** en cada fold se ajusta un `StandardScaler` usando
únicamente las ventanas de entrenamiento de ese fold, y se aplica a train/val.
Esto es crítico porque SMA_20/SMA_50/ATR/MACD están en USD y crecen 2015→2024:
sin normalizar, validación y test quedan fuera de la distribución de entrenamiento.
El scaler del mejor fold se guarda junto al checkpoint para usarse en test.

```
Total  ┃━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┃━━━━━━━┃
       ┃     Train + Val (85%)        ┃Test15%┃
Fold 0 ┃━━━━━━━━━━━━━━━━━┃Val┃        ┃       ┃
Fold 1 ┃━━━━━━━━━━━━━━━━━━━━┃Val┃     ┃       ┃
...    ┃                               ...     ┃
```

In [ ]:
# ── Hiperparámetros del modelo ───────────────────────────────────────────────
MODEL_HP = dict(
    price_input_size = N_FEATURES,
    sentiment_dim    = SENT_DIM,
    hidden_size      = 128,
    d_model          = 64,
    num_heads        = 4,
    num_lstm_layers  = 2,
    dropout          = 0.2,
)

TRAIN_HP = dict(
    lr           = 1e-3,
    weight_decay = 1e-5,
    w_t1         = 0.6,    # peso pérdida horizonte t+1
    w_t5         = 0.4,    # peso pérdida t+5 (Trainer normaliza su residuo por √5)
    epochs       = 100,
    patience     = 15,     # early stopping
    batch_size   = 32,
)

TEST_FRAC = 0.15
N_SPLITS  = 5

# ── Corte train_val / test ───────────────────────────────────────────────────
test_start, _ = final_test_split(N_SAMPLES, TEST_FRAC)
test_idx      = np.arange(test_start, N_SAMPLES)

print(f'Total muestras : {N_SAMPLES}')
print(f'Train+Val      : {test_start}  (índices 0..{test_start-1})')
print(f'Test           : {N_SAMPLES - test_start}  (índices {test_start}..{N_SAMPLES-1})')

# ── Walk-forward splits ──────────────────────────────────────────────────────
splits = walk_forward_splits(test_start, n_splits=N_SPLITS)
print(f'\nWalk-forward splits ({N_SPLITS} folds):')
for i, (tr, va) in enumerate(splits):
    print(f'  Fold {i}: train={len(tr):5d}  val={len(va):4d}')

# ── Verificar forward pass ───────────────────────────────────────────────────
_m  = HybridPredictiveModel(**MODEL_HP)
_ps = torch.randn(2, LOOKBACK, N_FEATURES)
_se = torch.randn(2, SENT_DIM)
_t1, _t5 = _m(_ps, _se)
print(f'\nForward pass OK: pred_t1={tuple(_t1.shape)}, pred_t5={tuple(_t5.shape)}')
print(f'Parámetros entrenables: {count_parameters(_m):,}')
del _m, _ps, _se, _t1, _t5

In [ ]:
fold_metrics   = []
fold_histories = []
best_val_loss  = float('inf')
best_scaler    = None

# Cerrar cualquier run activo antes de abrir uno nuevo (p.ej. tras re-ejecución de celda)
mlflow.end_run()

run_name  = f'Hybrid-h{MODEL_HP["hidden_size"]}-d{MODEL_HP["d_model"]}-wf{N_SPLITS}'
active_run = mlflow.start_run(run_name=run_name)

mlflow.log_params({**MODEL_HP, **TRAIN_HP,
                   'n_splits': N_SPLITS, 'test_frac': TEST_FRAC,
                   'lookback': LOOKBACK, 'ticker': TICKER,
                   'has_finbert': has_sentiment,
                   'feature_scaling': 'standard_per_fold'})

print(f'Run: {active_run.info.run_name}  |  ID: {active_run.info.run_id[:8]}\n')

for fold_idx, (train_idx, val_idx) in enumerate(splits):
    print(f'{"─"*60}')
    print(f'FOLD {fold_idx}  |  train={len(train_idx):5d}  val={len(val_idx):4d}')

    # Normalización sin leakage: scaler ajustado SOLO con ventanas de train de
    # este fold; transforma el array completo para que val use las mismas stats
    seqs_scaled, fold_scaler = scale_price_sequences(data['price_seqs'], train_idx)

    model = HybridPredictiveModel(**MODEL_HP).to(device)
    trainer = Trainer(
        model, device=device,
        lr=TRAIN_HP['lr'], weight_decay=TRAIN_HP['weight_decay'],
        w_t1=TRAIN_HP['w_t1'], w_t5=TRAIN_HP['w_t5'],
    )

    train_loader = make_dataloader(
        seqs_scaled, data['sentiments'], data['y_t1'], data['y_t5'],
        train_idx, batch_size=TRAIN_HP['batch_size'], shuffle=True,
    )
    val_loader = make_dataloader(
        seqs_scaled, data['sentiments'], data['y_t1'], data['y_t5'],
        val_idx, batch_size=TRAIN_HP['batch_size'],
    )

    history = trainer.fit(
        train_loader, val_loader,
        epochs=TRAIN_HP['epochs'],
        patience=TRAIN_HP['patience'],
        save_path=f'models/fold_{fold_idx}.pth',
    )
    fold_histories.append(history)

    # ── Métricas del fold (sobre validación) ────────────────────────────────
    # trainer.fit restaura los mejores pesos (early stopping), así que estas
    # métricas corresponden al mejor checkpoint del fold, no a la última época
    vp_t1, vp_t5 = predict(
        model, seqs_scaled[val_idx], data['sentiments'][val_idx], device=device
    )
    m_t1 = predictive_metrics(data['y_t1'][val_idx], vp_t1.flatten())
    m_t5 = predictive_metrics(data['y_t5'][val_idx], vp_t5.flatten())

    fold_best_loss = min(history['val_loss'])
    fold_metrics.append({
        'fold':    fold_idx,
        'val_loss': fold_best_loss,
        'rmse_t1': m_t1['RMSE'],
        'mae_t1':  m_t1['MAE'],
        'da_t1':   m_t1['Directional_Accuracy'],
        'rmse_t5': m_t5['RMSE'],
        'mae_t5':  m_t5['MAE'],
        'da_t5':   m_t5['Directional_Accuracy'],
    })

    mlflow.log_metrics({
        f'fold{fold_idx}_rmse_t1': m_t1['RMSE'],
        f'fold{fold_idx}_da_t1':   m_t1['Directional_Accuracy'],
        f'fold{fold_idx}_rmse_t5': m_t5['RMSE'],
        f'fold{fold_idx}_da_t5':   m_t5['Directional_Accuracy'],
    })

    # Guardar mejor checkpoint global (menor val_loss) + su scaler asociado
    if fold_best_loss < best_val_loss:
        best_val_loss = fold_best_loss
        best_scaler   = fold_scaler
        shutil.copy(f'models/fold_{fold_idx}.pth', 'models/hybrid_best.pth')
        joblib.dump(fold_scaler, 'models/hybrid_best_scaler.joblib')
        print(f'  ★ Nuevo mejor checkpoint (val_loss={best_val_loss:.6f})')

    print(f'  RMSE t+1={m_t1["RMSE"]:.4f}%  DA t+1={m_t1["Directional_Accuracy"]:.3f}')
    print(f'  RMSE t+5={m_t5["RMSE"]:.4f}%  DA t+5={m_t5["Directional_Accuracy"]:.3f}')

print(f'\n{"─"*60}')
print(f'Mejor checkpoint: models/hybrid_best.pth  (val_loss={best_val_loss:.6f})')
print(f'Scaler asociado : models/hybrid_best_scaler.joblib')

In [ ]:
# ── Tabla de resultados walk-forward ─────────────────────────────────────────
df_folds = pd.DataFrame(fold_metrics).set_index('fold')
df_folds.index = [f'Fold {i}' for i in df_folds.index]

print('Walk-Forward — Resultados por fold (val set):')
print(df_folds[['rmse_t1','mae_t1','da_t1','rmse_t5','mae_t5','da_t5']].to_string(
    float_format='{:.4f}'.format
))

print(f'\nPromedios walk-forward:')
for col in ['rmse_t1','da_t1','rmse_t5','da_t5']:
    print(f'  {col:10s}: {df_folds[col].mean():.4f} ± {df_folds[col].std():.4f}')

mlflow.log_metrics({
    'wf_avg_rmse_t1': df_folds['rmse_t1'].mean(),
    'wf_avg_da_t1':   df_folds['da_t1'].mean(),
    'wf_avg_rmse_t5': df_folds['rmse_t5'].mean(),
    'wf_avg_da_t5':   df_folds['da_t5'].mean(),
})

# ── Curvas de entrenamiento (último fold) ─────────────────────────────────────
plot_training_history(fold_histories[-1], save_path='curvas_entrenamiento.png')
print('Curvas del último fold guardadas: curvas_entrenamiento.png')

## 4. Evaluación en Test Set Out-of-Sample

Se evalúa el mejor checkpoint sobre el 15% final de los datos, nunca visto durante
ningún fold de entrenamiento ni validación. Se compara contra baselines: Naive (cero)
y Ridge Regression.

In [ ]:
from sklearn.linear_model import Ridge

# ── Cargar mejor checkpoint y su scaler ──────────────────────────────────────
best_model = HybridPredictiveModel(**MODEL_HP).to(device)
best_model.load_state_dict(torch.load('models/hybrid_best.pth', map_location=device))
best_model.eval()
best_scaler = joblib.load('models/hybrid_best_scaler.joblib')
print('Checkpoint cargado: models/hybrid_best.pth (+ scaler del mismo fold)')

# El test se transforma con el scaler del fold que produjo el mejor checkpoint:
# mismas estadísticas que vio el modelo en entrenamiento, sin leakage
seqs_test_scaled = transform_price_sequences(data['price_seqs'][test_idx], best_scaler)

# ── Predicciones en test ─────────────────────────────────────────────────────
preds_t1, preds_t5 = predict(
    best_model, seqs_test_scaled, data['sentiments'][test_idx], device=device
)
preds_t1 = preds_t1.flatten()
preds_t5 = preds_t5.flatten()

y_t1_test = data['y_t1'][test_idx]
y_t5_test = data['y_t5'][test_idx]

# ── Baselines ────────────────────────────────────────────────────────────────
# Ridge usa su propio scaler ajustado sobre todo train+val (nunca ve test)
train_all_idx = np.arange(0, test_start)
seqs_ridge, _ = scale_price_sequences(data['price_seqs'], train_all_idx)
X_flat_train = seqs_ridge[train_all_idx].reshape(len(train_all_idx), -1)
X_flat_test  = seqs_ridge[test_idx].reshape(len(test_idx), -1)

ridge_t1 = Ridge(alpha=1.0).fit(X_flat_train, data['y_t1'][train_all_idx])
ridge_t5 = Ridge(alpha=1.0).fit(X_flat_train, data['y_t5'][train_all_idx])

naive_pred    = np.zeros_like(y_t1_test)          # siempre predice retorno = 0
ridge_pred_t1 = ridge_t1.predict(X_flat_test)
ridge_pred_t5 = ridge_t5.predict(X_flat_test)

# ── Tabla comparativa ────────────────────────────────────────────────────────
def eval_row(y_true, y_pred, label):
    bt = long_short_strategy(y_true, y_pred)
    return {
        'Modelo': label,
        'RMSE (%)': rmse(y_true, y_pred),
        'MAE (%)':  mae(y_true, y_pred),
        'DA':       directional_accuracy(y_true, y_pred),
        'Sharpe':   bt['strategy_sharpe'],
        'MaxDD (%)': bt['strategy_max_drawdown'],
    }

rows = [
    eval_row(y_t1_test, naive_pred,    'Naive (cero)       t+1'),
    eval_row(y_t1_test, ridge_pred_t1, 'Ridge Regression   t+1'),
    eval_row(y_t1_test, preds_t1,      'HybridModel        t+1  ← este trabajo'),
    eval_row(y_t5_test, naive_pred,    'Naive (cero)       t+5'),
    eval_row(y_t5_test, ridge_pred_t5, 'Ridge Regression   t+5'),
    eval_row(y_t5_test, preds_t5,      'HybridModel        t+5  ← este trabajo'),
]
df_results = pd.DataFrame(rows).set_index('Modelo')

print(f'\n=== Test Out-of-Sample ({len(test_idx)} muestras) ===\n')
print(df_results.to_string(float_format='{:.4f}'.format))

# y_t5 es retorno acumulado de 5 días: su RMSE vive en escala ~√5×; el
# equivalente diario (RMSE/√5) es el comparable con t+1 y el target < 0.8%
rmse_t5_eq = rmse(y_t5_test, preds_t5) / np.sqrt(5)
print(f'\nNota: y_t5 = retorno ACUMULADO de 5 días (escala ~√5× la diaria).')
print(f'RMSE t+5 equivalente diario: {rmse_t5_eq:.4f}%')

# Loguear en MLflow
mlflow.log_metrics({
    'test_rmse_t1':  rmse(y_t1_test, preds_t1),
    'test_mae_t1':   mae(y_t1_test, preds_t1),
    'test_da_t1':    directional_accuracy(y_t1_test, preds_t1),
    'test_rmse_t5':  rmse(y_t5_test, preds_t5),
    'test_rmse_t5_daily_eq': rmse_t5_eq,
    'test_mae_t5':   mae(y_t5_test, preds_t5),
    'test_da_t5':    directional_accuracy(y_t5_test, preds_t5),
})

In [ ]:
plot_predictions(y_t1_test, preds_t1, horizon='t+1', save_path='predicciones_t1.png')
plot_predictions(y_t5_test, preds_t5, horizon='t+5', save_path='predicciones_t5.png')

## 5. Backtesting — Estrategia Long/Short

Señal: `pred > threshold` → largo (compra); `pred < −threshold` → corto (venta);
`|pred| ≤ threshold` → sin posición. Benchmark: Buy & Hold QQQ.

Incluye análisis de sensibilidad al `THRESHOLD` con métricas completas
(Sharpe, Sortino, MaxDD, retorno, win rate, exposición) por umbral, y una
recomendación automática: el umbral de máximo Sharpe entre los que mantienen
exposición ≥ 30%.

In [ ]:
THRESHOLD = 0.0   # umbral en % de retorno; la tabla de abajo ayuda a elegir el óptimo

bt_t1 = long_short_strategy(y_t1_test, preds_t1, threshold=THRESHOLD)
bt_t5 = long_short_strategy(y_t5_test, preds_t5, threshold=THRESHOLD)
bt_ridge_t1 = long_short_strategy(y_t1_test, ridge_pred_t1, threshold=THRESHOLD)

print(f'{"─"*65}')
print(f'{"Estrategia":<38} {"Sharpe":>7} {"Sortino":>8} {"MaxDD":>7} {"Trades":>7}')
print(f'{"─"*65}')

for label, bt in [
    ('Ridge Regression  t+1', bt_ridge_t1),
    ('HybridModel       t+1', bt_t1),
    ('HybridModel       t+5*', bt_t5),
]:
    print(
        f'{label:<38}'
        f'{bt["strategy_sharpe"]:>7.3f}'
        f'{bt["strategy_sortino"]:>8.3f}'
        f'{bt["strategy_max_drawdown"]:>7.2f}%'
        f'{bt["num_trades"]:>7d}'
    )

print(f'{"─"*65}')
print(f'Buy & Hold QQQ                         : Sharpe = {bt_t1["bh_sharpe"]:.3f}')
print('* t+5 compone retornos acumulados de 5 días SOLAPADOS con paso diario:')
print('  Sharpe/MaxDD quedan inflados — solo referencia direccional, no comparable.')

# Targets de tesis
print(f'\nTargets de la tesis:')
print(f'  RMSE < 0.8%   →  {"✓ CUMPLIDO" if rmse(y_t1_test, preds_t1) < 0.8 else "✗ aún no"} ({rmse(y_t1_test, preds_t1):.4f}%)')
print(f'  Sharpe > 0.5  →  {"✓ CUMPLIDO" if bt_t1["strategy_sharpe"] > 0.5 else "✗ aún no"} ({bt_t1["strategy_sharpe"]:.4f})')
print(f'  MaxDD > −20%  →  {"✓ CUMPLIDO" if bt_t1["strategy_max_drawdown"] > -20 else "✗ aún no"} ({bt_t1["strategy_max_drawdown"]:.2f}%)')

mlflow.log_metrics({
    'test_sharpe_t1':  bt_t1['strategy_sharpe'],
    'test_sortino_t1': bt_t1['strategy_sortino'],
    'test_mdd_t1':     bt_t1['strategy_max_drawdown'],
    'test_sharpe_t5':  bt_t5['strategy_sharpe'],
})

plot_cumulative_returns(bt_t1['strategy_returns'], bt_t1['bh_returns'],
                        save_path='backtesting_t1.png')

# ── Diagnóstico de predicciones ───────────────────────────────────────────────
print(f'\n=== Diagnóstico de predicciones (t+1) ===')
print(f'  min={preds_t1.min():.4f}%   max={preds_t1.max():.4f}%')
print(f'  mean={preds_t1.mean():.4f}%  std={preds_t1.std():.4f}%')
pct_pos = (preds_t1 > 0).mean()
print(f'  Pred > 0 (señal largo) : {pct_pos:.1%}')
print(f'  Pred < 0 (señal corto) : {1-pct_pos:.1%}')

# ── Selección de THRESHOLD: métricas completas por umbral ────────────────────
# THRESHOLD=0 opera todos los días (sin zona cash). Subirlo filtra señales
# débiles a costa de exposición. Esta tabla cuantifica el trade-off completo.
print(f'\n=== Sensibilidad al THRESHOLD (t+1) ===')
th_rows = []
for th in [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5]:
    bt = long_short_strategy(y_t1_test, preds_t1, threshold=th)
    signals = np.where(preds_t1 > th, 1.0, np.where(preds_t1 < -th, -1.0, 0.0))
    active  = signals != 0
    win_rate = float((bt['strategy_returns'][active] > 0).mean()) if active.any() else float('nan')
    th_rows.append({
        'THRESHOLD (%)': th,
        'Exposición':    float(active.mean()),
        'Sharpe':        bt['strategy_sharpe'],
        'Sortino':       bt['strategy_sortino'],
        'MaxDD (%)':     bt['strategy_max_drawdown'],
        'Retorno (%)':   bt['strategy_total_return'],
        'Win rate':      win_rate,
    })

df_th = pd.DataFrame(th_rows).set_index('THRESHOLD (%)')
print(df_th.to_string(float_format='{:.3f}'.format))

# Recomendación: máximo Sharpe entre umbrales con exposición >= 30%
# (con muy pocos días activos el Sharpe deja de ser estadísticamente fiable)
df_valid = df_th[df_th['Exposición'] >= 0.30]
th_opt = float(df_valid['Sharpe'].idxmax()) if len(df_valid) else 0.0
print(f'\n  → THRESHOLD recomendado: {th_opt:.2f}%')
print(f'    Sharpe={df_th.loc[th_opt, "Sharpe"]:.3f}  '
      f'MaxDD={df_th.loc[th_opt, "MaxDD (%)"]:.2f}%  '
      f'exposición={df_th.loc[th_opt, "Exposición"]:.1%}')

mlflow.log_metrics({
    'threshold_opt':          th_opt,
    'test_sharpe_t1_th_opt':  float(df_th.loc[th_opt, 'Sharpe']),
    'test_mdd_t1_th_opt':     float(df_th.loc[th_opt, 'MaxDD (%)']),
    'exposure_th_opt':        float(df_th.loc[th_opt, 'Exposición']),
})

In [ ]:
# ── Loguear artefactos y cerrar run ─────────────────────────────────────────
for fig_name in ['eda_overview.png', 'eda_acf_pacf.png', 'eda_correlacion.png',
                 'curvas_entrenamiento.png',
                 'predicciones_t1.png', 'predicciones_t5.png', 'backtesting_t1.png']:
    if Path(fig_name).exists():
        mlflow.log_artifact(fig_name, artifact_path='figures')

if Path('models/hybrid_best.pth').exists():
    mlflow.log_artifact('models/hybrid_best.pth', artifact_path='model')
if Path('models/hybrid_best_scaler.joblib').exists():
    mlflow.log_artifact('models/hybrid_best_scaler.joblib', artifact_path='model')

mlflow.end_run()
print('Run cerrado y subido a DagsHub.')
print(f'Ver en: https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow')

## 6. TimeGAN — Módulo Generativo *(opcional)*

Entrena el generador condicional `TimeGANGenerator + WassersteinCritic` (WGAN-GP)
para producir trayectorias sintéticas de 20 días condicionadas al sentimiento.

**Nota:** Con vectores de sentimiento = ceros (sin corpus FinBERT), el generador
aprende la distribución marginal de retornos. La conditioning real sobre sentimiento
se activa al integrar `finbert_embeddings.csv`.

Esta sección puede omitirse si el objetivo es solo el módulo predictivo.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader as TorchLoader

# ── Ventanas de 20 días de retornos para el generador ────────────────────────
# El generador trabaja con retornos univariados, no con los 10 features del
# módulo predictivo (los retornos ya son estacionarios; no requieren scaler)
returns_full = price_df['Daily_Return'].values.astype(np.float32)
SEQ_GAN      = 20

# Alineación: data['sentiments'][j] corresponde a price_df.index[LOOKBACK + j]
# (las primeras LOOKBACK filas no generan muestra en create_sequences).
# La ventana GAN i empieza en price_df[i] → su embedding es sentiments[i - LOOKBACK].
gan_windows, gan_sents = [], []
for i in range(len(returns_full) - SEQ_GAN):
    gan_windows.append(returns_full[i:i + SEQ_GAN])
    j = i - LOOKBACK
    if 0 <= j < len(data['sentiments']):
        sent_i = data['sentiments'][j]
    else:
        sent_i = np.zeros(SENT_DIM, dtype=np.float32)
    gan_sents.append(sent_i)

gan_windows = np.array(gan_windows, dtype=np.float32).reshape(-1, SEQ_GAN, 1)
gan_sents   = np.array(gan_sents,   dtype=np.float32)

# Mover TODO el dataset a VRAM de una vez (~10 MB total, cabe 1500x en los 15 GB disponibles)
gan_windows_gpu = torch.from_numpy(gan_windows).to(device)
gan_sents_gpu   = torch.from_numpy(gan_sents).to(device)

gan_ds      = TensorDataset(gan_windows_gpu, gan_sents_gpu)
gan_loader  = TorchLoader(gan_ds, batch_size=64, shuffle=True, drop_last=True)

print(f'Ventanas GAN   : {gan_windows.shape}  (n_windows, 20 días, 1 feature)')
print(f'Sentimiento GAN: {gan_sents.shape}')
print(f'Batches / época: {len(gan_loader)}')

In [ ]:
# Fix: CuDNN no soporta doble backward en RNNs (requerido por WGAN-GP gradient penalty)
# Sobrescribimos _gradient_penalty para deshabilitar CuDNN solo en ese forward pass
import src.train as _train_mod
import torch as _torch

def _gp_cudnn_fix(critic, real_seq, fake_seq, sentiment, device, lambda_gp=10.0):
    batch = real_seq.size(0)
    eps = _torch.rand(batch, 1, 1, device=device).expand_as(real_seq)
    interpolated = (eps * real_seq + (1 - eps) * fake_seq).requires_grad_(True)
    with _torch.backends.cudnn.flags(enabled=False):
        score_interp = critic(interpolated, sentiment)
    gradients = _torch.autograd.grad(
        outputs=score_interp, inputs=interpolated,
        grad_outputs=_torch.ones_like(score_interp),
        create_graph=True, retain_graph=True, only_inputs=True,
    )[0]
    grad_norm = gradients.view(batch, -1).norm(2, dim=1)
    return lambda_gp * ((grad_norm - 1.0) ** 2).mean()

_train_mod._gradient_penalty = _gp_cudnn_fix

NOISE_DIM  = 32
GAN_EPOCHS = 200   # aumentar a 500+ para mayor calidad distribucional

generator = TimeGANGenerator(
    noise_dim=NOISE_DIM, sentiment_dim=SENT_DIM,
    hidden_size=128, output_seq_len=SEQ_GAN, output_features=1, num_layers=2,
)
critic = WassersteinCritic(
    seq_features=1, sentiment_dim=SENT_DIM, hidden_size=128, num_layers=2,
)

gan_trainer = GANTrainer(
    generator=generator, critic=critic,
    noise_dim=NOISE_DIM, device=device,
    lr_gen=1e-4, lr_critic=1e-4,
    n_critic=5,      # 5 pasos critic : 1 paso generator (estándar WGAN-GP)
    lambda_gp=10.0,  # coeficiente gradient penalty
)

print(f'Generator params: {count_parameters(generator):,}')
print(f'Critic    params: {count_parameters(critic):,}')
print(f'Entrenando {GAN_EPOCHS} épocas  (n_critic=5, λ_gp=10.0)\n')

gan_history = gan_trainer.fit(
    gan_loader, epochs=GAN_EPOCHS, log_every=20,
    save_path='models/timegan_generator.pth',
)

# Curva Wasserstein distance
plt.figure(figsize=(10, 4))
plt.plot(gan_history['w_distance'], color='steelblue', label='W-distance ≈')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.title('Entrenamiento WGAN-GP — Wasserstein Distance', fontsize=12, fontweight='bold')
plt.xlabel('Época')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('wgan_training.png', dpi=150)
plt.show()
print('Generador guardado: models/timegan_generator.pth')

In [ ]:
# ── Generar escenarios sintéticos ────────────────────────────────────────────
# Condicionados a sentimiento neutro (ceros) — equivale a un día de mercado calmo
# Para stress-test: usar el embedding del crash COVID (2020-03-16) si disponible
neutral_sentiment = np.zeros((1, SENT_DIM), dtype=np.float32)
scenarios = generate_scenarios(
    generator, neutral_sentiment, noise_dim=NOISE_DIM, n_scenarios=100, device=device
)  # (100, 20, 1)

real_windows_sample = gan_windows[:100, :, 0]   # 100 ventanas reales

# ── Métricas generativas ─────────────────────────────────────────────────────
gen_m = generative_metrics(real_windows_sample, scenarios[:, :, 0])

print('=== Métricas Generativas ===')
print(f'  Wasserstein Distance  : {gen_m["wasserstein_distance"]:.4f}  (0 = perfecto)')
print(f'  Vol. clustering real  : {gen_m["real_vol_clustering"]:.4f}')
print(f'  Vol. clustering gen.  : {gen_m["fake_vol_clustering"]:.4f}  (deben ser similares)')
print(f'  Leverage real         : {gen_m["real_leverage"]:.4f}')
print(f'  Leverage generado     : {gen_m["fake_leverage"]:.4f}')
print(f'  Kurtosis real         : {gen_m["real_excess_kurtosis"]:.4f}')
print(f'  Kurtosis generada     : {gen_m["fake_excess_kurtosis"]:.4f}')

plot_generated_scenarios(real_windows_sample, scenarios[:, :, 0],
                         n_scenarios=20, save_path='escenarios_sinteticos.png')

## 7. Resumen Final

In [ ]:
print('=' * 65)
print('   RESUMEN — PROTOTIPO FUNCIONAL SISTEMA HÍBRIDO QQQ')
print('=' * 65)
print(f'  Ticker   : {TICKER}  |  Período: {START_DATE} → {END_DATE}')
print(f'  Features : {N_FEATURES} (9 técnicos + VIX)  |  FinBERT: {"REAL" if has_sentiment else "CEROS (sin corpus)"}')
print(f'  Modelo   : HybridPredictiveModel  |  Params: {count_parameters(best_model):,}')
print(f'  Escalado : StandardScaler por fold (sin leakage)')
print(f'  Targets  : t+1 retorno diario  |  t+5 retorno ACUMULADO de 5 días')
print()
print(f'  Walk-Forward ({N_SPLITS} folds) — promedio:')
print(f'    RMSE t+1 = {df_folds["rmse_t1"].mean():.4f}% ± {df_folds["rmse_t1"].std():.4f}%')
print(f'    DA   t+1 = {df_folds["da_t1"].mean():.3f}')
print(f'    RMSE t+5 = {df_folds["rmse_t5"].mean():.4f}% ± {df_folds["rmse_t5"].std():.4f}%  (equiv. diario: {df_folds["rmse_t5"].mean()/np.sqrt(5):.4f}%)')
print(f'    DA   t+5 = {df_folds["da_t5"].mean():.3f}')
print()
print(f'  Test Out-of-Sample ({len(test_idx)} muestras):')
print(f'    RMSE t+1 = {rmse(y_t1_test, preds_t1):.4f}%')
print(f'    DA   t+1 = {directional_accuracy(y_t1_test, preds_t1):.3f}')
print(f'    Sharpe t+1 = {bt_t1["strategy_sharpe"]:.3f}  |  MaxDD = {bt_t1["strategy_max_drawdown"]:.2f}%')
print(f'    RMSE t+5 = {rmse(y_t5_test, preds_t5):.4f}%  (equiv. diario: {rmse(y_t5_test, preds_t5)/np.sqrt(5):.4f}%)')
print(f'    DA   t+5 = {directional_accuracy(y_t5_test, preds_t5):.3f}')
print(f'    THRESHOLD recomendado = {th_opt:.2f}%  (Sharpe={df_th.loc[th_opt, "Sharpe"]:.3f})')
print()
print(f'  Targets de la tesis:')
rmse_ok   = rmse(y_t1_test, preds_t1) < 0.8
sharpe_ok = bt_t1['strategy_sharpe'] > 0.5
mdd_ok    = bt_t1['strategy_max_drawdown'] > -20
print(f'    RMSE < 0.8%    → {"✓" if rmse_ok   else "✗"}')
print(f'    Sharpe > 0.5   → {"✓" if sharpe_ok else "✗"}')
print(f'    MaxDD > −20%   → {"✓" if mdd_ok    else "✗"}')
print()
print(f'  Estado FinBERT:')
print(f'    {"✓ Activo" if has_sentiment else "✗ Sin corpus — siguiente paso crítico"}')
print()
print(f'  PRÓXIMOS PASOS:')
print(f'    1. Construir corpus FinBERT: FNSPID (Kaggle) + Tiingo API → finbert_embeddings.csv')
print(f'    2. Reentrenar con embeddings reales y comparar RMSE vs baseline de ceros')
print(f'    3. Ablation study: rama precio sola vs precio + sentimiento')
print(f'    4. Aumentar GAN_EPOCHS (500+) para mayor calidad distribucional TimeGAN')
print('=' * 65)